In [ ]:
"""Convert Philippines AWS (DAVIS + LUFFT) monthly CSVs into AirNow-format
surface obs for MELODIES-MONET.

"""

import glob
import os

import numpy as np
import pandas as pd

# --------------------------------------------------------------------------
ROOT = "/glade/campaign/acom/acom-weather/emmons/ASIAAQ_obs/AWS_Philippines"
OUT_DIR = "/glade/u/home/lcthompson/mm/MELODIES-MONET/docs/examples/ungridded_support/unstructured_grid_read_uxarray/asiaaq_cs_06082026/preprocessing/philippines_data"                      # where to write outputs
PHT_OFFSET_HOURS = 8               # Philippine Standard Time = UTC+8 (no DST)
ASSUME_LOCAL = True                # True: CSV times are PHT; False: already UTC

# siteid to (latitude, longitude, site_name)
DAVIS_META = {
    "MOIP":  (14.636, 121.077, "Loyola Heights, Quezon City (Manila Observatory)"),
    "mw002": (14.673, 121.079, "Brgy. Holy Spirit, Quezon City (Caltex)"),
    "mw003": (14.634, 121.059, "Brgy. Anonas, Quezon City (Caltex)"),
    "mw006": (14.583, 121.097, "Manggahan Floodway, Pasig City (Caltex)"),
    "mw007": (14.657, 120.992, "U. Plata, Caloocan City (Caltex)"),
    "mw010": (14.626, 120.989, "Dimasalang cor. Blumentritt Sta. Cruz, Manila (Caltex)"),
    "mw011": (14.382, 121.045, "Tunasan, Muntinlupa (Caltex)"),
    "mw017": (14.668, 120.943, "M. Naval St., Navotas"),
    "mw028": (14.608, 121.104, "Felix Avenue Bgy. Dela Paz, Pasig City (Caltex)"),
}

LUFFT_META = {
    "WPF001": (14.6262,     121.08432,   "SM Marikina"),
    "WPF002": (14.62555,    121.12025,   "SM Masinag"),
    "WPF003": (14.58427,    121.0775,    "SM Hypermarket Pasig"),
    "WPF004": (14.58912,    121.05706,   "SM Megamall"),
    "WPF005": (14.61898,    121.05533,   "SM Cubao"),
    "WPF006": (14.60457,    121.01935,   "SM Sta. Mesa"),
    "WPF007": (14.59047,    120.98374,   "SM Manila"),
    "WPF008": (14.61827,    120.98647,   "SM San Lazaro"),
    "WPF009": (14.54596,    121.05344,   "SM Aura"),
    "WPF010": (14.70844,    121.03912,   "SM Novaliches"),
    "WPF011": (14.56526,    121.14072,   "SM Taytay"),
    "WPF012": (14.5313,     121.15564,   "SM Angono"),
    "WPF013": (14.43378,    121.01066,   "SM Southmall"),
    "WPF014": (14.44909,    120.98085,   "SM Center Las Pinas"),
    "WPF015": (14.37817,    121.04648,   "SM Center Muntinlupa"),
    "WPF016": (14.45777,    121.0339,    "SM BF Homes"),
    "WPF017": (14.48704,    121.04409,   "SM Bicutan"),
    "WPF018": (14.40938,    120.85818,   "SM Rosario"),
    "WPF019": (14.75407,    120.9564,    "SM Marilao"),
    "WPF020": (14.68557,    120.97792,   "SM Valenzuela"),
    "WPF021": (15.0525,     120.7022,    "SM PAMPANGA"),
    "WPF022": (15.02778,    120.6922,    "SM DOWNTOWN SAN FERNANDO"),
    "WPF023": (15.1697,     120.5807,    "SM CLARK"),
    "WPF024": (14.8261,     120.283,     "SM OLONGAPO"),
    "WPF025": (15.4784,     120.594,     "SM TARLAC"),
    "WPF026": (15.8784,     120.6006,    "SM ROSALES"),
    "WPF027": (14.31314,    121.09909,   "SM Sta. Rosa"),
    "WPF028": (14.20408,    121.15644,   "SM Calamba"),
    "WPF029": (14.0715517,  121.3022166, "SM San Pablo"),
    "WPF030": (13.9541903,  121.1638831, "SM Lipa"),
    "WPF032": (13.94045941, 121.6250839, "SM Lucena"),
    "WPF033": (18.52651078, 120.7043031, "North Wind - Ilocos Norte"),
    "WPF035": (16.40907693, 120.5997576, "SM Baguio"),
    "WPF037": (13.62134123, 123.1908434, "SM Naga"),
    "WPF039": (14.30124311, 120.9568682, "SM Dasmarinas"),
    "WPF042": (14.445,      120.9508,    "SM Bacoor"),
    "WPF989243": (14.6356,  121.0938,    "Marikina River Park"),
}

# Network under ROOT
NETWORKS = {
    "DAVIS": dict(subdir="DAVIS", meta=DAVIS_META),
    "LUFFT": dict(subdir="LUFFT", meta=LUFFT_META),
}

COLMAP = {
    "rain":     "precip_accum",       # accumulated rainfall
    "temp":     "temperature",
    "wspeed":   "wind_speed",
    "wdir":     "wind_direction",
    "pressure": "surface_pressure",
    "hum":      "relative_humidity",
    "dewp":     "dewpoint",
    "srad":     "solar_radiation",
}

UNITS = {
    "precip_accum":      "mm",
    "temperature":       "degC",
    "wind_speed":        "m/s",
    "wind_direction":    "deg",
    "surface_pressure":  "hPa",
    "relative_humidity": "%",
    "dewpoint":          "degC",
    "solar_radiation":   "W/m^2",
}

# Values that mean "missing/invalid". -999 (incl. WDir during calm winds) ->
# NaN. read_csv also coerces the string forms.
NA_STRINGS = ["NA", "N/A", "-999", "-999.0", ""]
MISSING_NUMERIC = (-999.0,)

DATE_FMT = "%m/%d/%Y %H:%M"
# --------------------------------------------------------------------------

def _build_time(df):
    """Return (local_time Series, [time_columns_to_exclude]).

    """
    lc = {c.lower(): c for c in df.columns}

    date_col = lc.get("date")
    time_col = lc.get("time")
    if date_col and time_col:
        combo = (
            df[date_col].astype(str).str.strip()
            + " "
            + df[time_col].astype(str).str.strip()
        )
        t = pd.to_datetime(combo, format=DATE_FMT, errors="coerce")
        if t.notna().mean() <= 0.5:  # fall back to flexible parsing
            t = pd.to_datetime(combo, errors="coerce")
        if t.notna().mean() < 0.5:
            raise ValueError(
                f"Date+Time failed to parse (e.g. {combo.iloc[0]!r}); "
                f"check DATE_FMT={DATE_FMT!r} and the Date/Time columns."
            )
        return t, [date_col, time_col]

    # single combined datetime column
    for hint in ("datetime", "timestamp", "date_time", "obs_time", "date", "time"):
        if hint in lc:
            t = pd.to_datetime(df[lc[hint]], errors="coerce")
            if t.notna().mean() > 0.8:
                return t, [lc[hint]]

    # last resort: first parseable column
    for c in df.columns:
        t = pd.to_datetime(df[c], errors="coerce")
        if t.notna().mean() > 0.8:
            return t, [c]
    raise ValueError(f"No timestamp column(s) found among {list(df.columns)}")


def _read_one_csv(path, siteid, lat, lon, site, network):

    df = pd.read_csv(path, na_values=NA_STRINGS, keep_default_na=True)
    df.columns = [str(c).strip() for c in df.columns]
    if df.empty:
        return None

    t_local, time_cols = _build_time(df)

    # UTC time
    if ASSUME_LOCAL:
        time_utc = t_local - pd.to_timedelta(PHT_OFFSET_HOURS, unit="h")
        utcoffset = PHT_OFFSET_HOURS
        time_local = t_local
    else:
        time_utc = t_local
        utcoffset = PHT_OFFSET_HOURS
        time_local = t_local + pd.to_timedelta(PHT_OFFSET_HOURS, unit="h")

    # every other column that is numeric becomes a `variable`
    value_cols = [c for c in df.columns if c not in time_cols]
    long_rows = []
    for c in value_cols:
        vals = pd.to_numeric(df[c], errors="coerce")
        vals = vals.replace(list(MISSING_NUMERIC), np.nan)  # -999 -> NaN
        if vals.notna().sum() == 0:
            continue  # skip text/flag columns
        var = COLMAP.get(c.strip().lower(), c)
        long_rows.append(
            pd.DataFrame(
                {
                    "time": time_utc.values,
                    "time_local": time_local.values,
                    "utcoffset": utcoffset,
                    "siteid": siteid,
                    "site": site,
                    "latitude": lat,
                    "longitude": lon,
                    "variable": var,
                    "units": UNITS.get(var, "unknown"),
                    "obs": vals.values,
                    "network": network,
                }
            )
        )
    if not long_rows:
        return None
    out = pd.concat(long_rows, ignore_index=True)
    return out.dropna(subset=["time", "obs"])


def load_network(network, subdir, meta):
    rows = []
    net_dir = os.path.join(ROOT, subdir)
    for siteid, (lat, lon, site) in meta.items():
        files = sorted(glob.glob(os.path.join(net_dir, siteid, f"{siteid}_*.csv")))
        if not files:
            print(f"  [{network}] {siteid}: no files found")
            continue
        for f in files:
            try:
                r = _read_one_csv(f, siteid, lat, lon, site, network)
                if r is not None:
                    rows.append(r)
            except Exception as e:  # noqa: BLE001
                print(f"  [{network}] failed {os.path.basename(f)}: {e}")
    if not rows:
        return pd.DataFrame()
    print(f"  [{network}] read {len(rows)} files")
    return pd.concat(rows, ignore_index=True)


def to_airnow_long(df):

    for col in ["cmsa_name", "msa_code", "msa_name", "state_name", "epa_region"]:
        df[col] = np.nan
    df["siteid"] = df["siteid"].astype(str)
    cols = [
        "time", "siteid", "site", "utcoffset", "variable", "units", "obs",
        "time_local", "latitude", "longitude", "cmsa_name", "msa_code",
        "msa_name", "state_name", "epa_region", "network",
    ]
    return df[cols].sort_values(["siteid", "variable", "time"]).reset_index(drop=True)


def long_to_wide(df_long):
    
    idx = ["time", "siteid", "site", "latitude", "longitude",
           "utcoffset", "time_local", "network"]
    wide = (
        df_long.pivot_table(index=idx, columns="variable", values="obs", aggfunc="mean")
        .reset_index()
    )
    wide.columns.name = None
    return wide


def wide_to_netcdf(df_wide, path):

    meta_cols = ["site", "latitude", "longitude", "utcoffset", "network", "time_local"]
    var_cols = [c for c in df_wide.columns
                if c not in (["time", "siteid"] + meta_cols)]

    df = df_wide.copy()
    sites = df["siteid"].drop_duplicates().tolist()
    df["x"] = df["siteid"].map({s: i for i, s in enumerate(sites)})

    ds = df.set_index(["x", "time"])[var_cols].to_xarray()  # dims (x, time)

    # WD and accum precip need to watch this as it wouldnt be corrct
    ds = ds.resample(time="1h").mean()

    site_meta = df.drop_duplicates("x").set_index("x").sort_index().reindex(ds["x"].values)
    ds = ds.assign_coords(
        # cast to plain str (not pandas StringDtype) so NetCDF encoding works
        siteid=("x", site_meta["siteid"].astype(str).to_numpy(dtype=object)),
        latitude=("x", site_meta["latitude"].to_numpy(dtype="float64")),
        longitude=("x", site_meta["longitude"].to_numpy(dtype="float64")),
        utcoffset=("x", site_meta["utcoffset"].to_numpy(dtype="float64")),
    )
    # time_local = time + utcoffset, as (x, time), so MM's
    # ts_select_time: 'time_local' has a column to index on.
    off = pd.to_timedelta(ds["utcoffset"].values, unit="h")   # (x,)
    t = pd.DatetimeIndex(ds["time"].values)                   # (time,)
    ds["time_local"] = (("x", "time"), t.values[None, :] + off.values[:, None])
    ds["time_local"] = ds["time_local"].transpose("time", "x")
    ds.to_netcdf(path)
    return ds

In [ ]:
all_long = []
for network, cfg in NETWORKS.items():
    print(f"Reading {network} ...")
    df = load_network(network, cfg["subdir"], cfg["meta"])
    if not df.empty:
        all_long.append(df)
if not all_long:
    raise SystemExit("No data read -- check ROOT and directory layout.")

long = to_airnow_long(pd.concat(all_long, ignore_index=True))
wide = long_to_wide(long)

print(f"\nTotal long rows: {len(long):,}")
print("Variables found:", sorted(long['variable'].unique()))
print("Sites:", long['siteid'].nunique(),
      "| time range:", long['time'].min(), "to", long['time'].max())

long_csv = os.path.join(OUT_DIR, "philippines_aws_airnow_long.csv")
wide_csv = os.path.join(OUT_DIR, "philippines_aws_airnow_wide.csv")
nc = os.path.join(OUT_DIR, "philippines_aws_airnow.nc")
long.to_csv(long_csv, index=False)
wide.to_csv(wide_csv, index=False)
wide_to_netcdf(wide, nc)
print(f"\nwrote:\n  {long_csv}\n  {wide_csv}\n  {nc}")